In [ ]:
import spread_cell_mesh_generation as mesh_gen

radiusArray=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
             0.5, 0.25, 0.5, 0.25, 
             0.0, 0.0, 0.0, 0.0, 0.0]
pitchArray=[5.0, 2.5, 1.0, 5.0, 2.5, 1.0,
            5.0, 2.5, 5.0, 2.5,
            0.0, 0.0, 0.0, 0.0, 0.0]
heightArray= [1.0, 1.0, 1.0, 3.0, 3.0, 3.0,
              1.0, 1.0, 3.0, 3.0,
              0.0, 0.0, 0.0, 0.0, 0.0]
cellRadArray=[20.25, 18.52, 16.55, 19.93, 18.04, 15.39,
              20.01, 17.45, 18.06, 17.64,
              22.48, 18.08, 15.39, 14.18, 12.33]

idx = 9
cell_mesh, facet_markers, cell_markers, substrate_markers, u_nuc, a_nuc = (
                 mesh_gen.create_3dcell(contactRad=cellRadArray[idx],
                                        hEdge=0.5, hInnerEdge=0.5, hNP=0.2,
                                        nanopillars=[radiusArray[idx], heightArray[idx], pitchArray[idx]],
                                        sym_fraction=0.125, no_nuc=True))
print('done')

In [ ]:
import spread_cell_mesh_generation as mesh_gen
import numpy as np
import dolfin as d
import pathlib
import matplotlib.pyplot as plt

radiusArray=[0.515/2, 1.30/2]
pitchArray=[3.0, 3.0]
heightArray= [[0.0, 0.5, 1.5, 3.0, 3.18, 3.18],
              [0.0, 0.5, 1.5, 3.0, 4.0, 4.7]]
cellAreaArray=[[500, 600, 700, 700, 700, 900],
              [500, 600, 700, 700, 700, 900]]
tVec = [0, 10, 60, 120, 300, 480]
size = ["small", "large"]
shape_coords_stored = [[None]*6]*2

for i in range(len(size)):
    plt.figure()
    for j in [1]:#range(len(tVec)):
        cur_folder = pathlib.Path(f"{size[i]}_t{tVec[j]}")
        cur_folder.mkdir(exist_ok=True)
        cellRad = np.sqrt(cellAreaArray[i][j]/np.pi)
        nanopillars = [radiusArray[i], heightArray[i][j], pitchArray[i]]
        rValsOuter, zValsOuter, rValsInner, zValsInner, innerParam, u_nuc, rScale, outParam = mesh_gen.get_shape_coords(
                                            cellRad, nanopillars, 0.0, no_nuc=True)
        shape_coords = [rValsOuter, zValsOuter, rValsInner, zValsInner, innerParam, u_nuc]
        mesh, mf2, mf3, s, u, a = mesh_gen.create_3dcell(contactRad=cellRad, shape_coords=shape_coords,
                                                         hEdge=0.5, hInnerEdge=0.5, hNP=0.2,
                                                         nanopillars=nanopillars,
                                                         sym_fraction=0.125, no_nuc=True)
        
        d.File(str(cur_folder / "mesh.pvd")) << mesh
        d.File(str(cur_folder / "mf2.pvd")) << mf2
        d.File(str(cur_folder / "mf3.pvd")) << mf3

        hCur = heightArray[i][j]
        zValsOuter = zValsOuter - hCur
        if heightArray[i][j] == 0:
            rSubstrate = np.array([0])
            zSubstrate = np.array([0])
        else:
            rLoc = np.arange(0.0, cellRad+0.2+radiusArray[i], pitchArray[i])
            rSubstrate = np.array([])
            zSubstrate = np.array([])
            for k in range(len(rLoc)):
                rmax = rLoc[k] + radiusArray[i] + 0.2
                if k == 0:
                    rCur = [0, rmax, rmax]
                    zCur = [0, 0, -hCur]
                elif rmax > cellRad:
                    rmin = rLoc[k] - radiusArray[i] - 0.2
                    rCur = [rmin, rmin, cellRad]
                    zCur = [-hCur, 0.0, 0.0]
                    lastIdx = np.nonzero(zValsOuter > 0.0)[0][-1]
                    rValsOuter = rValsOuter[:lastIdx+1]
                    zValsOuter = zValsOuter[:lastIdx+1]
                else:
                    rmin = rLoc[k] - radiusArray[i] - 0.2
                    rCur = [rmin, rmin, rmax, rmax]
                    zCur = [-hCur, 0.0, 0.0, -hCur]
                rSubstrate = np.concatenate((rSubstrate, rCur))
                zSubstrate = np.concatenate((zSubstrate, zCur))
        rValsOuter = np.concatenate((rValsOuter, rSubstrate[::-1]))
        zValsOuter = np.concatenate((zValsOuter, zSubstrate[::-1]))
        plt.plot(rValsOuter, zValsOuter)
        cur_vol = d.assemble(1.0*d.Measure("dx",mesh))
        cur_sa = d.assemble(1.0*d.Measure("ds",mesh))
        shape_coords_stored[i][j] = [rValsOuter, zValsOuter, outParam, cur_vol, cur_sa]
        print(f'Done with {size[i]} nanopillars, t = {tVec[j]} min, volume = {cur_vol}')

In [ ]:
import meshio
gmsh_file_name = "test.msh"

mesh_in = meshio.read(gmsh_file_name)
meshio.write("test.ply", mesh_in)